In [209]:
# importando as bibliotecas necessárias

import pandas as pd
import numpy as np
import re

In [210]:
# importando os dados

vendas = pd.read_csv("halloween_vendas.csv", encoding="utf-8", sep=",")

## Análise Exploratória 

In [211]:
# mostrando as primeiras 3 linhas do dataframe

vendas.head(3)

,pedido_id,data_venda,produto,categoria,canal_venda,cidade,quantidade,preco_unitario,custo_unitario,desconto_pct,status_pedido,publico,tamanho,data_entrega
0,844,2025-11-21 18:02:35,Fantasia de vampiro,Fantasia,Loja física,Curitiba,3,132.5,72.08,15,Cancelado,Adulto,G,NaN
1,378,2024-10-29 18:41:26,Fantasia de bruxa,FANTASIA!!!,Loja física,Maringá,3,110.0,57.00,5,Concluído,Adulto,M,2024-11-03 18:41:26
2,953,2025-09-05 21:16:09,Fantasia de bruxa,Fantasia,Web,Maringá,2,116.6,60.42,5,Concluído,Adulto,M,2025-09-10 21:16:09


In [212]:
vendas.shape

(1500, 14)

In [213]:
vendas.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   pedido_id       1500 non-null   int64  
 1   data_venda      1500 non-null   str    
 2   produto         1500 non-null   str    
 3   categoria       1482 non-null   str    
 4   canal_venda     1500 non-null   str    
 5   cidade          1500 non-null   str    
 6   quantidade      1500 non-null   int64  
 7   preco_unitario  1488 non-null   float64
 8   custo_unitario  1493 non-null   float64
 9   desconto_pct    1500 non-null   int64  
 10  status_pedido   1500 non-null   str    
 11  publico         1500 non-null   str    
 12  tamanho         1500 non-null   str    
 13  data_entrega    1357 non-null   str    
dtypes: float64(2), int64(3), str(9)
memory usage: 164.2 KB


Verificando os valores únicos de cada coluna

In [214]:
# Verificando os valores presentes na coluna produto
# Para identificar nomes diferentes para o mesmo produto

vendas["produto"].value_counts()

produto
Fantasia de vampiro     272
Chapeu de bruxa         272
Fantasia de bruxa       255
Fantasia de abobora     240
Mascara assustadora     237
Fantasia de fantasma    224
Name: count, dtype: int64

In [215]:
# Verificando os valores presentes na coluna categoria
# Para identificar nomes diferentes para a mesma categoria

# Para esta coluna precisaremos realizar um tratamento nos valores
# pois as categorias fantasia e acessorio foram escritas de diversas maneiras

vendas["categoria"].value_counts()

categoria
Fantasia            892
Acessorio           459
 Fantasia            48
  FANTASIA!!!        39
 Acessorio           28
  ACESSORIO!!!       16
Name: count, dtype: int64

In [216]:
# Verificando os valores presentes na coluna canal_venda
# Para identificar nomes diferentes para o mesmo canal de venda

# Para esta coluna precisaremos realizar um tratamento nos valores
# pois os canais de venda estão com nomes divergentes dos canais padrão

vendas["canal_venda"].value_counts()

canal_venda
Web              646
Loja física      535
Mercado          274
 web              20
 loja física      16
 mercado           9
Name: count, dtype: int64

In [217]:
# Verificando os valores presentes na coluna cidade
# Para identificar nomes diferentes para a mesma cidade

vendas["cidade"].value_counts()

cidade
Londrina    396
Cascavel    378
Maringá     375
Curitiba    351
Name: count, dtype: int64

In [218]:
# Verificando os valores presentes na coluna status_pedido
# Para identificar nomes diferentes para o mesmo status de pedido

vendas["status_pedido"].value_counts()

status_pedido
Concluído    1345
Cancelado     109
Devolvido      46
Name: count, dtype: int64

In [219]:
# Verificando os valores presentes na coluna publico
# Para identificar nomes diferentes para o mesmo tipo de público

vendas["publico"].value_counts()

publico
Adulto      527
Todos       509
Infantil    464
Name: count, dtype: int64

In [220]:
# Verificando os valores presentes na coluna tamanho
# Para identificar nomes diferentes para o mesmo tamanho de produto

vendas["tamanho"].value_counts()

tamanho
Unico    509
M        479
G        272
P        240
Name: count, dtype: int64

Precisaremos tratar os valores das colunas "categoria" e "canal_venda"

In [221]:
# Verificar quais colunas possuem valores NaN

vendas.isna().sum()

pedido_id           0
data_venda          0
produto             0
categoria          18
canal_venda         0
cidade              0
quantidade          0
preco_unitario     12
custo_unitario      7
desconto_pct        0
status_pedido       0
publico             0
tamanho             0
data_entrega      143
dtype: int64

Em 4 colunas temos linhas com valores nulos, teremos que analisar o contexto de cada informação ausente e verificar a necessidade de tratamento

## INICIO TRATAMENTO DATASET


### Validação e Limpeza de Dados Ausentes e Numéricos

In [222]:
vendas_tratado = vendas.copy()

In [223]:
vendas_tratado.shape

(1500, 14)

In [224]:
# Preencher categorias vazias com a string `"sem categoria"`

vendas_tratado["categoria"] = vendas_tratado["categoria"].fillna("sem categoria")

In [225]:
# Contar o total de pedidos afetados por problemas

preco_filtro = (pd.isna(vendas_tratado["preco_unitario"])) | (vendas_tratado["preco_unitario"] < 0)

print(f"Valores inválidos (NaN ou menor/igual a zero) na coluna 'preco_unitario': {vendas_tratado.loc[preco_filtro].shape[0]}")


qtd_filtro = (pd.isna(vendas_tratado["quantidade"])) | (vendas_tratado["quantidade"] <= 0)

print(f"\nValores inválidos (NaN ou menor/igual a zero) na coluna 'quantidade': {vendas_tratado.loc[qtd_filtro].shape[0]}")


custo_filtro = pd.isna(vendas_tratado["custo_unitario"])

print(f"\nQuantidade de valores NaN na coluna 'custo_unitario': {vendas_tratado.loc[custo_filtro].shape[0]}")

Valores inválidos (NaN ou menor/igual a zero) na coluna 'preco_unitario': 12

Valores inválidos (NaN ou menor/igual a zero) na coluna 'quantidade': 7

Quantidade de valores NaN na coluna 'custo_unitario': 7


In [226]:
# criado uma função para validar alguns valores númericos do dataset

def validar_linha (linha):
  """ Função para validar as linhas de um dataframe.
        Retorna 1 se a linha for INVÁLIDA e 0 se for VÁLIDA.
  """

  # Verifica se o valor da coluna quantidade é valido
  if (pd.isna(linha["quantidade"])) | (linha["quantidade"] <= 0):
    return 1

  # Verifica se o valor da coluna preco_unitario é valido
  if (pd.isna(linha["preco_unitario"])) | (linha["preco_unitario"] < 0):
    return 1

  # Verifica se o valor da coluna "custo_unitario" é NaN.
  # No documento não foi específicado uma regra de negócio para essa coluna
  # então será feito somente a verificação de nulos
  if (pd.isna(linha["custo_unitario"])):
    return 1

  else: return 0


In [227]:
# Aplicando a função para criar uma nova coluna no dataframe
# Servindo para rotular quais linhas possuem algum valor numérico inválido

vendas_tratado["numerico_invalido"] = vendas_tratado.apply(validar_linha, axis=1)

In [228]:
# Verificando se ficou correto a rotulação das linhas

vendas_tratado.loc[vendas_tratado["numerico_invalido"] == 0].isna().sum()


pedido_id              0
data_venda             0
produto                0
categoria              0
canal_venda            0
cidade                 0
quantidade             0
preco_unitario         0
custo_unitario         0
desconto_pct           0
status_pedido          0
publico                0
tamanho                0
data_entrega         139
numerico_invalido      0
dtype: int64

In [229]:
# Contando o total de pedidos que possuem algum valor numérico inválido
# Evitando contar mais de uma vez a linha que contenha mais de um problema
ocorrencias = vendas_tratado.loc[vendas_tratado["numerico_invalido"] == 1].shape[0]

print(f"\nO total de pedidos únicos afetados por problemas é de {ocorrencias} ocorrências")


O total de pedidos únicos afetados por problemas é de 26 ocorrências


### Padronização de Strings e Regex

In [230]:
# Criação da função normalizar_string
# .strip - remove os espaços
# .lower - deixa as letras em minúsculo
# re.sub r"[^\w\s]" - mantém apenas os caracteres alfanuméricos e espaços

def normalizar_string(texto : str) -> str:
    """
    Função para remover espaços desnecessários, deixar todas as letras minúsculas e remover pontuações de uma string
    """

    # Deixa em minúsculo e remove espaços nas pontas
    texto = texto.strip().lower()

    # Remove tudo o que NÃO for palavra ou espaço
    texto = re.sub(r"[^\w\s]", "", texto)

    return texto

In [231]:
# Aplica a função na coluna desejada

vendas_tratado["categoria"] = vendas_tratado["categoria"].apply(normalizar_string)

In [232]:
# Verifica se funcionou corretamente

vendas_tratado["categoria"].value_counts()

categoria
fantasia         979
acessorio        503
sem categoria     18
Name: count, dtype: int64

In [233]:
# Formatando os valores da coluna canal_venda
# .strip() remove os espaços
# .capitalize() deixa a primeira letra maiúscula

vendas_tratado["canal_venda"] = vendas_tratado["canal_venda"].str.strip().str.capitalize()

In [234]:
vendas_tratado["canal_venda"].value_counts()

canal_venda
Web            666
Loja física    551
Mercado        283
Name: count, dtype: int64

In [235]:
# Categorias vazias que foram preenchidas
qtd_vazias = vendas_tratado["categoria"].loc[vendas_tratado["categoria"] == "sem categoria"].count()

# Categorias não vazias que foram alteradas
qtd_alteradas = vendas["categoria"].loc[(vendas["categoria"] != "fantasia") & (vendas["categoria"] != "acessorio")].count()

print(f"Quantidade de categorias vazias preenchidas: {qtd_vazias}")
print(f"\nQuantidade de categorias não vazias alteradas: {qtd_alteradas}")
print(f"\nQuantidade total de categorias preenchidas e padronizadas: {qtd_vazias + qtd_alteradas}")

Quantidade de categorias vazias preenchidas: 18

Quantidade de categorias não vazias alteradas: 1482

Quantidade total de categorias preenchidas e padronizadas: 1500


### Formatação Temporal (Datetime)

In [236]:
# Primeiro renomeamos a coluna original para manter a identificação

vendas_tratado = vendas_tratado.rename(columns={"data_venda" : "data_venda_original"})

In [237]:
# Criaremos uma nova coluna para armazenar as datas no formato datetime

vendas_tratado["data_venda_tratado"] = pd.to_datetime(vendas_tratado["data_venda_original"], format="%Y-%m-%d %H:%M:%S", errors="coerce")

In [238]:
# Agora conforme pedido criaremos uma coluna para representação da data no formato brasileiro

vendas_tratado["data_venda"] = vendas_tratado["data_venda_tratado"].dt.strftime("%d/%m/%Y")

In [239]:
# Verificamos se as novas colunas criadas ficaram no formato correto

vendas_tratado.dtypes

pedido_id                       int64
data_venda_original               str
produto                           str
categoria                         str
canal_venda                       str
cidade                            str
quantidade                      int64
preco_unitario                float64
custo_unitario                float64
desconto_pct                    int64
status_pedido                     str
publico                           str
tamanho                           str
data_entrega                      str
numerico_invalido               int64
data_venda_tratado     datetime64[us]
data_venda                        str
dtype: object

In [240]:
def validar_datas(linha):
  """
  \nFunção para validar se as datas presentes na linha do dataframe possuem alguma data inválida.
  \nRetorna 1 se tiver data inválida e 0 se as datas estiverem corretas
  \nVerifica os campos 'data_venda_tratado' e 'data_entrega'
  """
  if pd.isna(linha["data_venda_tratado"]):
    return 1
  if pd.isna(linha["data_entrega"]):
    return 1
  else: return 0

In [241]:
# Aplica no dataframe a função validar_datas
# Criando a nova coluna is_data_invalida que tem 1 caso a data seja inválida
# e tem 0 caso a linha possua apenas datas válidas

vendas_tratado["is_data_invalida"] = vendas_tratado.apply(validar_datas, axis=1)

### Lógica de Regra de Negócio (Filtros e Validação de Entrega)

In [242]:
# Verificar se todos os pedidos que não possuem registro da data de entrega
# possuem o status de pedido como "cancelado"

vendas_tratado["status_pedido"].loc[pd.isna(vendas_tratado["data_entrega"])].value_counts()

status_pedido
Cancelado    109
Concluído     34
Name: count, dtype: int64

In [243]:
# Número de pedidos sem data de entrega e com o status como Cancelado

contagem = vendas_tratado.loc[(pd.isna(vendas_tratado["data_entrega"])) & (vendas_tratado["status_pedido"] == "Cancelado")].shape[0]

print(f"Existem {contagem} pedidos sem data de entrega e com o status Cancelado")

Existem 109 pedidos sem data de entrega e com o status Cancelado


In [244]:
# Número de pedidos sem data de entrega mas com o status como Concluído

contagem = vendas_tratado.loc[(pd.isna(vendas_tratado["data_entrega"])) & (vendas_tratado["status_pedido"] != "Cancelado")].shape[0]

print(f"Existem {contagem} pedidos sem data de entrega e com o outro status que não é Cancelado")

Existem 34 pedidos sem data de entrega e com o outro status que não é Cancelado


In [245]:
# Exibir até 3 exemplos do segundo grupo (pedidos sem entrega e com outro status)

print(f"3 exemplos de linhas que não possuem data de entrega mas que o status não é Cancelado: \n")

vendas_tratado.loc[(pd.isna(vendas_tratado["data_entrega"])) & (vendas_tratado["status_pedido"] != "Cancelado")].head(3)

3 exemplos de linhas que não possuem data de entrega mas que o status não é Cancelado: 



,pedido_id,data_venda_original,produto,categoria,canal_venda,cidade,quantidade,preco_unitario,custo_unitario,desconto_pct,status_pedido,publico,tamanho,data_entrega,numerico_invalido,data_venda_tratado,data_venda,is_data_invalida
49,1170,2025-08-19 22:20:19,Fantasia de abobora,fantasia,Web,Maringá,2,90.1,46.64,0,Concluído,Infantil,P,NaN,0,2025-08-19 22:20:19,19/08/2025,1
117,990,2025-10-13 20:00:16,Fantasia de fantasma,fantasia,Web,Londrina,2,79.5,41.34,0,Concluído,Infantil,M,NaN,0,2025-10-13 20:00:16,13/10/2025,1
152,740,data_invalida,Fantasia de abobora,fantasia,Web,Curitiba,2,85.0,44.00,25,Concluído,Infantil,P,NaN,0,NaT,NaN,1


Nas informações acima podemos observar que existem 34 registros sem data de entrega, porém com status do pedido concluído. Então fica descartada a hipótese de que a falta de dados na coluna data_entrega é consequência somente de pedidos que não foi realizado a entrega, pois existem registros de produtos que foram entregues porém sem o registro da data.

Uma segunda hipótese para justificar registros com status concluído porém sem data de entrega, seria que os registros estavam duplicados e a data de entrega foi gravada em apenas uma das linhas duplicadas

In [246]:
# Verificando se há linhas duplicadas no dataset original
#

print(f"Quantidade de linhas duplicadas no dataset: {vendas.duplicated().sum()}")

Quantidade de linhas duplicadas no dataset: 0


In [247]:
# Verificando se há mais de uma linha com o mesmo pedido_id

print(f"Número de registros com o id repetido: {vendas.duplicated(subset=["pedido_id"]).sum()}")

Número de registros com o id repetido: 0


In [248]:
print(f"Número de registros com id único: {vendas["pedido_id"].nunique()}")

Número de registros com id único: 1500


Após a análise das informações acima podemos descartar a segunda hipótese também, pois nos dados não há registros com pedido_id duplicados

In [249]:
# Exibir o total geral de pedidos cancelados

total_cancelados = vendas_tratado.loc[vendas_tratado["status_pedido"] == "Cancelado"].shape[0]

print(f"Existem no total {total_cancelados} pedidos com o status de Cancelado")

Existem no total 109 pedidos com o status de Cancelado


### Análise Agregada (Métricas de Negócio)

In [250]:
vendas_tratado["canal_venda"].value_counts(ascending=True)

canal_venda
Mercado        283
Loja física    551
Web            666
Name: count, dtype: int64

In [251]:
# Contar pedidos por canal
#
mercado = vendas_tratado["canal_venda"].loc[vendas_tratado["canal_venda"] == "Mercado"].count()

loja_fisica = vendas_tratado["canal_venda"].loc[vendas_tratado["canal_venda"] == "Loja física"].count()

web = vendas_tratado["canal_venda"].loc[vendas_tratado["canal_venda"] == "Web"].count()

filtro = ((vendas_tratado["canal_venda"] != "Mercado")
          & (vendas_tratado["canal_venda"] != "Loja física")
          & (vendas_tratado["canal_venda"] != "Web")
          ) | (vendas_tratado["canal_venda"].isna())

outros_canais = vendas_tratado["canal_venda"].loc[filtro].count()

# Exibindo os resultados
print(f"Pedidos em Mercado: {mercado}")
print(f"\nPedidos em Loja física: {loja_fisica}")
print(f"\nPedidos via Web: {web}")
print(f"\nQuantidade de pedidos em outros canais ou em canais inválidos: {outros_canais}")

Pedidos em Mercado: 283

Pedidos em Loja física: 551

Pedidos via Web: 666

Quantidade de pedidos em outros canais ou em canais inválidos: 0


In [252]:
# Contar pedidos por ano-mês utilizando apenas registros com `data_venda` válida
# incluindo todos os status e informando exclusões
# sem agrupar apenas por mês para não misturar anos

print("Quantidade de pedidos por mês em cada ano:")

resultado = vendas_tratado.groupby([
    vendas_tratado["data_venda_tratado"].dt.year,
    vendas_tratado["data_venda_tratado"].dt.month
])["quantidade"].count().rename_axis(["Ano", "Mês"])

# exibe removendo os decimais e renomeando as colunas
display(resultado.reset_index().astype({"Ano": int, "Mês": int}))

print(f"\n{vendas_tratado["data_venda_tratado"].isna().sum()} Pedidos não foram contabilizados pois a data de venda é inválida")

Quantidade de pedidos por mês em cada ano:


,Ano,Mês,quantidade
0,2024,8,77
1,2024,9,104
2,2024,10,393
3,2024,11,174
4,2025,8,62
5,2025,9,94
6,2025,10,410
7,2025,11,182



4 Pedidos não foram contabilizados pois a data de venda é inválida


In [253]:
# Descoberta por canal

In [254]:
vendas_tratado["canal_venda"].value_counts()

canal_venda
Web            666
Loja física    551
Mercado        283
Name: count, dtype: int64

In [255]:
web_concluido = vendas_tratado["pedido_id"].loc[(vendas_tratado["canal_venda"] == "Web") & (vendas_tratado["status_pedido"] == "Concluído")].count()
web_cancelado = vendas_tratado["pedido_id"].loc[(vendas_tratado["canal_venda"] == "Web") & (vendas_tratado["status_pedido"] != "Concluído")].count()

taxa_problema = 100 * (web_cancelado / (web_concluido + web_cancelado))
taxa_sucesso = 100 * (web_concluido / (web_concluido + web_cancelado))

print(f"{web_concluido} pedidos via WEB foram realizados com sucesso")
print(f"\n{taxa_sucesso:.2f}% dos pedidos via web foram concluidos")
print(f"\n{taxa_problema:.2f}% dos pedidos via web foram cancelados ou devolvidos")

569 pedidos via WEB foram realizados com sucesso

85.44% dos pedidos via web foram concluidos

14.56% dos pedidos via web foram cancelados ou devolvidos


In [256]:
vendas_tratado["status_pedido"].loc[vendas_tratado["canal_venda"] == "Loja física"].value_counts()

status_pedido
Concluído    508
Cancelado     31
Devolvido     12
Name: count, dtype: int64

In [257]:
loja_concluido = vendas_tratado["pedido_id"].loc[(vendas_tratado["canal_venda"] == "Loja física") & (vendas_tratado["status_pedido"] == "Concluído")].count()
loja_cancelado = vendas_tratado["pedido_id"].loc[(vendas_tratado["canal_venda"] == "Loja física") & (vendas_tratado["status_pedido"] != "Concluído")].count()

taxa_problema = 100 * (loja_cancelado / (loja_concluido + loja_cancelado))
taxa_sucesso = 100 * (loja_concluido / (loja_concluido + loja_cancelado))

print(f"{loja_concluido} pedidos em lojas físicas foram realizados com sucesso")
print(f"\n{taxa_sucesso:.2f}% dos pedidos em lojas físicas foram concluidos")
print(f"\n{taxa_problema:.2f}% dos pedidos em lojas físicas foram cancelados ou devolvidos")

508 pedidos em lojas físicas foram realizados com sucesso

92.20% dos pedidos em lojas físicas foram concluidos

7.80% dos pedidos em lojas físicas foram cancelados ou devolvidos


In [258]:
vendas_tratado["status_pedido"].loc[vendas_tratado["canal_venda"] == "Mercado"].value_counts()

status_pedido
Concluído    268
Cancelado      8
Devolvido      7
Name: count, dtype: int64

In [259]:
mercado_concluido = vendas_tratado["pedido_id"].loc[(vendas_tratado["canal_venda"] == "Mercado") & (vendas_tratado["status_pedido"] == "Concluído")].count()
mercado_cancelado = vendas_tratado["pedido_id"].loc[(vendas_tratado["canal_venda"] == "Mercado") & (vendas_tratado["status_pedido"] != "Concluído")].count()

taxa_problema = 100 * (mercado_cancelado / (mercado_concluido + mercado_cancelado))
taxa_sucesso = 100 * (mercado_concluido / (mercado_concluido + mercado_cancelado))

print(f"{mercado_concluido} pedidos em mercados foram realizados com sucesso")
print(f"\n{taxa_sucesso:.2f}% dos pedidos em mercados foram concluidos")
print(f"\n{taxa_problema:.2f}% dos pedidos em mercados foram cancelados ou devolvidos")

268 pedidos em mercados foram realizados com sucesso

94.70% dos pedidos em mercados foram concluidos

5.30% dos pedidos em mercados foram cancelados ou devolvidos


In [260]:
web_total = web_concluido + web_cancelado
loja_total = loja_concluido + loja_cancelado
mercado_total = mercado_concluido + mercado_cancelado
pedidos_total = vendas_tratado["canal_venda"].count()
pedidos_concluido = vendas_tratado["pedido_id"].loc[(vendas_tratado["status_pedido"] == "Concluído")].count()
pedidos_cancelado = vendas_tratado["pedido_id"].loc[(vendas_tratado["status_pedido"] != "Concluído")].count()

print(f"{(100 * web_total/pedidos_total):.2f}% pedidos foram feitos no canal Web")
print(f"{(100 * loja_total/pedidos_total):.2f}% pedidos foram feitos no canal Loja física")
print(f"{(100 * mercado_total/pedidos_total):.2f}% pedidos foram feitos no canal Mercado")

44.40% pedidos foram feitos no canal Web
36.73% pedidos foram feitos no canal Loja física
18.87% pedidos foram feitos no canal Mercado


In [261]:
print(f"% de pedidos cancelados/devolvidos por canal:\n")

print(f"{(100 * web_cancelado / pedidos_cancelado):.2f}% dos pedidos cancelados/devolvidos são do canal Web")
print(f"{(100 * loja_cancelado / pedidos_cancelado):.2f}% dos pedidos cancelados/devolvidos são de Lojas físicas")
print(f"{(100 * mercado_cancelado / pedidos_cancelado):.2f}% dos pedidos cancelados/devolvidos são de Mercado")

% de pedidos cancelados/devolvidos por canal:

62.58% dos pedidos cancelados/devolvidos são do canal Web
27.74% dos pedidos cancelados/devolvidos são de Lojas físicas
9.68% dos pedidos cancelados/devolvidos são de Mercado


In [262]:
print(f"% de pedidos concluídos por canal:\n")

print(f"{(100 * web_concluido / pedidos_concluido):.2f}% dos pedidos concluídos são do canal Web")
print(f"{(100 * loja_concluido / pedidos_concluido):.2f}% dos pedidos concluídos são de Lojas físicas")
print(f"{(100 * mercado_concluido / pedidos_concluido):.2f}% dos pedidos concluídos são de Mercado")

% de pedidos concluídos por canal:

42.30% dos pedidos concluídos são do canal Web
37.77% dos pedidos concluídos são de Lojas físicas
19.93% dos pedidos concluídos são de Mercado


In [263]:
# Vende mais só que é mais cancelado

x = (100 * web_total/pedidos_total) - (100 * loja_total/pedidos_total)
y = (100 * web_cancelado / pedidos_cancelado) - (100 * loja_cancelado / pedidos_cancelado)

print(f"O canal Web é o que mais possui pedidos registrados.")
print(f"Ele possui {x:.2f}% pedidos a mais que Lojas físicas, que é o segundo canal com mais pedidos.")
print(f"Entretanto possui {y:.2f}% mais cancelamentos/devoluções que as Lojas físicas")

O canal Web é o que mais possui pedidos registrados.
Ele possui 7.67% pedidos a mais que Lojas físicas, que é o segundo canal com mais pedidos.
Entretanto possui 34.84% mais cancelamentos/devoluções que as Lojas físicas


In [264]:
# Criando uma nova coluna que contém o valor do faturamento por pedido

vendas_tratado["faturamento_pedido"] = vendas_tratado["preco_unitario"] * vendas_tratado["quantidade"]

In [265]:
faturamento_web = vendas_tratado["faturamento_pedido"].loc[(vendas_tratado["canal_venda"] == "Web") & (vendas_tratado["status_pedido"] == "Concluído")].mean()
faturamento_loja = vendas_tratado["faturamento_pedido"].loc[(vendas_tratado["canal_venda"] == "Loja física") & (vendas_tratado["status_pedido"] == "Concluído")].mean()
faturamento_mercado = vendas_tratado["faturamento_pedido"].loc[(vendas_tratado["canal_venda"] == "Mercado") & (vendas_tratado["status_pedido"] == "Concluído")].mean()

In [266]:
print(f"Média de faturamento por pedido em cada canal:\n")
print(f"Web: R${faturamento_web:.2f}")
print(f"Loja física: R${faturamento_loja:.2f}")
print(f"Mercado: R${faturamento_mercado:.2f}")

Média de faturamento por pedido em cada canal:

Web: R$177.15
Loja física: R$152.87
Mercado: R$203.45


In [267]:
print(f"Produtos com mais pedidos por canal: \n")
print(f"    Web     | {vendas_tratado["produto"].loc[vendas_tratado["canal_venda"] == "Web"].mode()[0]}")
print(f"Loja física | {vendas_tratado["produto"].loc[vendas_tratado["canal_venda"] == "Loja física"].mode()[0]}")
print(f"  Mercado   | {vendas_tratado["produto"].loc[vendas_tratado["canal_venda"] == "Mercado"].mode()[0]}")

Produtos com mais pedidos por canal: 

    Web     | Fantasia de vampiro
Loja física | Fantasia de bruxa
  Mercado   | Chapeu de bruxa


In [268]:
# Descoberta por período

In [269]:
# vendas por mês em 2024

vendas_mes_2024 = vendas_tratado["data_venda_tratado"].dt.month.loc[(vendas_tratado["data_venda_tratado"].dt.year == 2024)].value_counts()

display(vendas_mes_2024)

data_venda_tratado
10.0    393
11.0    174
9.0     104
8.0      77
Name: count, dtype: int64

In [270]:
print(f"As vendas do mês de Outubro representam {(100 * vendas_mes_2024[10]/vendas_mes_2024.sum()):.2f}% do total de vendas no ano de 2024")

As vendas do mês de Outubro representam 52.54% do total de vendas no ano de 2024


In [271]:
# Vendas por mês em 2025

vendas_mes_2025 = vendas_tratado["data_venda_tratado"].dt.month.loc[(vendas_tratado["data_venda_tratado"].dt.year == 2025)].value_counts()
display(vendas_mes_2025)

data_venda_tratado
10.0    410
11.0    182
9.0      94
8.0      62
Name: count, dtype: int64

In [272]:
print(f"As vendas do mês de Outubro representam {(100 * vendas_mes_2025[10]/vendas_mes_2025.sum()):.2f}% do total de vendas no ano de 2025")

As vendas do mês de Outubro representam 54.81% do total de vendas no ano de 2025


In [273]:
# Em ambos os anos o mês que mais vendeu foi o mês 10

In [274]:
# Analisando os pedidos do mês de novembro

vendas_tratado["data_venda_tratado"].dt.day.loc[ 
                                                (vendas_tratado["data_venda_tratado"].dt.month == 11)
                                                ].value_counts().sort_index()

data_venda_tratado
1.0     27
2.0     24
3.0     31
4.0     25
5.0     24
6.0     20
7.0     15
8.0     12
9.0     14
10.0    11
11.0    16
12.0    12
13.0    12
14.0     9
15.0    11
16.0    10
17.0    13
18.0     7
19.0     6
20.0    12
21.0     6
22.0     5
23.0     4
24.0     4
25.0     3
26.0     4
27.0     5
28.0     2
29.0     4
30.0     8
Name: count, dtype: int64

- Foi constatado que o mês de outubro sozinho representa mais de 50% dos pedidos feitos nos 4 meses de registro por ano

- Também foi possível observar que mesmo após o halloween, durante o mês de novembro, o número de pedidos ainda é considerável. Com a quantidade de pedidos se concentrando nos primeiros dias do mês e diminuindo ao longo do tempo, onde nos últimos dias do mês a quantidade de pedidos é consideravelmente baixa.

In [275]:
vendas_tratado["data_venda_tratado"].dt.hour.loc[(vendas_tratado["data_venda_tratado"].dt.month == 10) & (vendas_tratado["data_venda_tratado"].dt.year == 2024)].value_counts()

data_venda_tratado
18.0    63
19.0    52
22.0    42
20.0    38
21.0    28
10.0    24
9.0     23
14.0    21
12.0    20
11.0    19
17.0    18
13.0    17
15.0    16
16.0    12
Name: count, dtype: int64

In [276]:
# Separando em duas variaveis a quantidade de vendas em 2024 por horário

vendas_2024_noite = vendas_tratado["data_venda_tratado"].dt.hour.loc[(vendas_tratado["data_venda_tratado"].dt.year == 2024) &
                                                 ((vendas_tratado["data_venda_tratado"].dt.hour >= 18) & (vendas_tratado["data_venda_tratado"].dt.hour <= 23))
                                                 ].value_counts().sum()


vendas_2024_dia = vendas_tratado["data_venda_tratado"].dt.hour.loc[(vendas_tratado["data_venda_tratado"].dt.year == 2024)&
                                                 ((vendas_tratado["data_venda_tratado"].dt.hour >= 6) & (vendas_tratado["data_venda_tratado"].dt.hour < 18))
                                                 ].value_counts().sum()

In [277]:
print(f"Em 2024 as vendas após às 18:00 representaram {100 * vendas_2024_noite / (vendas_2024_noite + vendas_2024_dia):.2f}% do total de vendas")

Em 2024 as vendas após às 18:00 representaram 53.74% do total de vendas


In [278]:
# Separando em duas variaveis a quantidade de vendas em 2025 por horário

vendas_2025_noite = vendas_tratado["data_venda_tratado"].dt.hour.loc[(vendas_tratado["data_venda_tratado"].dt.year == 2025) &
                                                 ((vendas_tratado["data_venda_tratado"].dt.hour >= 18) & (vendas_tratado["data_venda_tratado"].dt.hour <= 23))
                                                 ].value_counts().sum()

vendas_2025_dia = vendas_tratado["data_venda_tratado"].dt.hour.loc[(vendas_tratado["data_venda_tratado"].dt.year == 2025)&
                                                 ((vendas_tratado["data_venda_tratado"].dt.hour >= 6) & (vendas_tratado["data_venda_tratado"].dt.hour < 18))
                                                 ].value_counts().sum()

In [279]:
print(f"Em 2025 as vendas após às 18:00 representaram {100 * vendas_2025_noite / (vendas_2025_noite + vendas_2025_dia):.2f}% do total de vendas")

Em 2025 as vendas após às 18:00 representaram 56.28% do total de vendas


 - Em 4 horas, a partir das 18:00, são feitos mais da metade do total de pedidos. Ficando a outra metade dos pedidos distribuídos pelas outras 9 horas do dia.

In [280]:
# total de linhas lidas

print(f"Total de linhas lidas: {vendas_tratado.shape[0]}")

Total de linhas lidas: 1500


In [281]:
# categorias preenchidas de padronizadas

filtro_categoria = ((vendas["categoria"] != "fantasia") & (vendas["categoria"] != "acessorio"))

print(f"Foram preenchidas e padronizadas {vendas["pedido_id"].loc[filtro_categoria].count()} linhas da coluna categoria")

Foram preenchidas e padronizadas 1500 linhas da coluna categoria


In [282]:
# Problemas númericos

print(f"Foram rotulados {vendas_tratado["numerico_invalido"].sum()} registros que possuem problemas numéricos")

Foram rotulados 26 registros que possuem problemas numéricos


In [283]:
# Datas inválidas

print(f"Foram rotulados {vendas_tratado["is_data_invalida"].sum()} registros que possuem data inválida")

Foram rotulados 143 registros que possuem data inválida


In [284]:
# total de cancelados

print(f"{vendas_tratado["pedido_id"].loc[vendas_tratado["status_pedido"] == "Cancelado"].count()} pedidos possuem o status de Cancelado")

109 pedidos possuem o status de Cancelado


In [285]:
# Sem data de entrega com status cancelado

print(f"{vendas_tratado["pedido_id"].loc[(pd.isna(vendas_tratado["data_entrega"])) & 
                                         (vendas_tratado["status_pedido"] == "Cancelado")].count()} pedidos possuem data de entrega inválida e status Cancelado")

109 pedidos possuem data de entrega inválida e status Cancelado


In [286]:
# Sem data de entrega com status concluido

print(f"{vendas_tratado["pedido_id"].loc[(pd.isna(vendas_tratado["data_entrega"])) & 
                                         (vendas_tratado["status_pedido"] == "Concluído")].count()} pedidos possuem data de entrega inválida e status Concluído")

34 pedidos possuem data de entrega inválida e status Concluído
